In [2]:
import pandas as pd
import numpy as np

## Data Transformation Explained

To predict if an artist plays in a specific year (e.g., 2026), we need to restructure our data.
Currently, we have **one row per artist** with a list of years played.
We need **one row per artist-year combination** (the "Long Format").

**Transformation Steps:**
1.  **Parse "Years Played":** Convert the string `"2022, 2024"` into binary columns: `played_2022`, `played_2023`, etc.
2.  **Clean Agency Data:** Identify which artists are managed by "Insomniac" (the festival organizer).
3.  **Create "Lag Features":** For every target year (e.g., 2025), looking backwards:
    *   Did they play 1 year ago? (2024)
    *   Did they play 2 years ago? (2023)
    *   How many total times have they played *before* this year?
4.  **Target Variable:** Did they actually play in the target year? (0 or 1)

In [3]:
# Load datasets
try:
    df_stats = pd.read_csv('../data/main/COMPLETE_edc_artist_and_stats.csv')
    df_residency = pd.read_csv('../data/extract/2026_vegas_recidency.csv')
    print("Data loaded successfully.")
    display(df_stats.head(3))
except FileNotFoundError:
    print("Files not found. Please check the paths.")

Data loaded successfully.


,artist,followers,streams,playlists,playlist reach,charts,shazams,videos,views,dj supports,total_appearances,years_played,agency
0,1080p,6,5986,1,641,0,17,45,9800,0,1,2024,insomniac
1,a shade of black,0,15100,0,11000,7,0,0,0,0,1,2024,NaN
2,aaron k,1,2163,0,0,1,4,2,0,0,2,"2024, 2025",insomniac


In [ ]:
# Parse "Years Played" into Binary Columns
# ---------------------------------------------------------

HISTORY_YEARS = [2022, 2023, 2024, 2025]

def parse_years(x):
    if pd.isna(x):
        return []
    # Clean the string: remove quotes, split by comma, convert to int
    return [int(y.strip()) for y in str(x).replace('"', '').split(',') if y.strip().isdigit()]

# Apply the parser
df_stats['years_played_list'] = df_stats['years_played'].apply(parse_years)

# Create binary columns for each year
for year in HISTORY_YEARS:
    col_name = f'played_{year}'
    df_stats[col_name] = df_stats['years_played_list'].apply(lambda x: 1 if year in x else 0)

# Check our work
cols_to_show = ['artist', 'years_played'] + [f'played_{y}' for y in HISTORY_YEARS]
display(df_stats[cols_to_show].head())

# 2. Clean Agency Data (Insomniac Flag)
# ---------------------------------------------------------
# Create a simple binary flag for the main organizer
df_stats['is_insomniac'] = df_stats['agency'].fillna('').str.lower().str.contains('insomniac').astype(int)

# Log Scale for Followers (Handles huge range 100 vs 10M)
df_stats['log_followers'] = np.log1p(df_stats['followers'].fillna(0))

print(f"Insomniac Artists found: {df_stats['is_insomniac'].sum()}")
df_stats.head()

,artist,years_played,played_2022,played_2023,played_2024,played_2025
0,1080p,2024,0,0,1,0
1,a shade of black,2024,0,0,1,0
2,aaron k,"2024, 2025",0,0,1,1
3,abana,"2022, 2023, 2024",1,1,1,0
4,d. zeledon,2024,0,0,1,0


Insomniac Artists found: 573


,artist,followers,streams,playlists,playlist reach,charts,shazams,videos,views,dj supports,total_appearances,years_played,agency,years_played_list,played_2022,played_2023,played_2024,played_2025,is_insomniac,log_followers
0,1080p,6,5986,1,641,0,17,45,9800,0,1,2024,insomniac,[2024],0,0,1,0,1,1.945910
1,a shade of black,0,15100,0,11000,7,0,0,0,0,1,2024,NaN,[2024],0,0,1,0,0,0.000000
2,aaron k,1,2163,0,0,1,4,2,0,0,2,"2024, 2025",insomniac,"[2024, 2025]",0,0,1,1,1,0.693147
3,abana,7710,468000,175,4220000,60,15300,110,1890000,93,3,"2022, 2023, 2024",insomniac,"[2022, 2023, 2024]",1,1,1,0,1,8.950403
4,d. zeledon,5776,69600,25,218000,18,764,31,9672,58,1,2024,insomniac,[2024],0,0,1,0,1,8.661640


In [7]:
# Create Training Set (Simulate 2025)
# ---------------------------------------------------------
# We pretend we are in late 2024 trying to predict the 2025 lineup.
# This allows us to have a "Ground Truth" (who actually played in 2025) to train our model.

training_data = []
target_year = 2025

for idx, row in df_stats.iterrows():
    # INPUTS: What did we know BEFORE 2025?
    played_prev_year = row['played_2024']    # Played last year?
    played_2_years_ago = row['played_2023']  # Played 2 years ago?
    played_3_years_ago = row['played_2022']  # Played 3 years ago?
    
    # Feature: Total past appearances (up to 2024)
    total_past_appearances = row['played_2024'] + row['played_2023'] + row['played_2022']
    
    # Feature: Consecutive years played recently (burnout indicator)
    consecutive = 0
    if played_prev_year == 1:
        consecutive = 1
        if played_2_years_ago == 1:
            consecutive = 2
            
    # TARGET: Did they ACTUALLY play in 2025?
    played_target_year = row['played_2025']
    
    # Combine everything into a training row
    training_data.append({
        'artist': row['artist'],
        'played_prev_year': played_prev_year,
        'played_2_years_ago': played_2_years_ago,
        'played_3_years_ago': played_3_years_ago,
        'total_past_appearances': total_past_appearances,
        'consecutive_years': consecutive,
        'is_insomniac': row['is_insomniac'],
        'log_followers': row['log_followers'],
        'streams': row['streams'], # Raw popularity metric
        'target_played_2025': played_target_year # PREDICTION TARGET
    })

df_train = pd.DataFrame(training_data)

print(f"Training Data Created: {df_train.shape[0]} samples")
display(df_train.head())

Training Data Created: 973 samples


,artist,played_prev_year,played_2_years_ago,played_3_years_ago,total_past_appearances,consecutive_years,is_insomniac,log_followers,streams,target_played_2025
0,1080p,1,0,0,1,1,1,1.945910,5986,0
1,a shade of black,1,0,0,1,1,0,0.000000,15100,0
2,aaron k,1,0,0,1,1,1,0.693147,2163,1
3,abana,1,1,1,3,2,1,8.950403,468000,0
4,d. zeledon,1,0,0,1,1,1,8.661640,69600,0


In [8]:
# Train the Model (Random Forest)
# ---------------------------------------------------------
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score

# Define Features and Target
features = ['played_prev_year', 'played_2_years_ago', 'played_3_years_ago', 
            'total_past_appearances', 'consecutive_years', 
            'is_insomniac', 'log_followers', 'streams']
target = 'target_played_2025'

# Initialize Model
# n_estimators=100: Build 100 decision trees
# class_weight='balanced': Handle the fact that most artists DON'T played (imbalanced data)
model = RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42, class_weight='balanced')

# Fit Model on the training data
model.fit(df_train[features], df_train[target])

print("Model Trained Successfully!")

# Check Feature Importance (What did the model learn?)
importances = pd.DataFrame({'feature': features, 'importance': model.feature_importances_})
importances = importances.sort_values('importance', ascending=False)
display(importances)

Model Trained Successfully!


,feature,importance
3,total_past_appearances,0.374430
6,log_followers,0.156863
7,streams,0.133016
2,played_3_years_ago,0.131635
1,played_2_years_ago,0.075161
4,consecutive_years,0.050238
0,played_prev_year,0.049130
5,is_insomniac,0.029527


In [11]:
# Predict 2026 (The Future)
# ---------------------------------------------------------
# Now we gather the data as it stands TODAY (for the 2026 festival).

data_2026 = []

for idx, row in df_stats.iterrows():
    # INPUTS for 2026 Prediction (shifted forward by 1 year)
    played_prev_year = row['played_2025']    # 2025 (Last year relative to 2026)
    played_2_years_ago = row['played_2024']  # 2024
    played_3_years_ago = row['played_2023']  # 2023
    
    # Total past (2022-2025)
    total_past_appearances = row['played_2025'] + row['played_2024'] + row['played_2023'] + row['played_2022']
    
    consecutive = 0
    if played_prev_year == 1:
        consecutive = 1
        if played_2_years_ago == 1:
            consecutive = 2
    
    data_2026.append({
        'artist': row['artist'],
        'played_prev_year': played_prev_year,
        'played_2_years_ago': played_2_years_ago,
        'played_3_years_ago': played_3_years_ago,
        'total_past_appearances': total_past_appearances,
        'consecutive_years': consecutive,
        'is_insomniac': row['is_insomniac'],
        'log_followers': row['log_followers'],
        'streams': row['streams']
    })

df_2026 = pd.DataFrame(data_2026)

# Make Probability Predictions (0 to 1)
# predict_proba returns [prob_0, prob_1] -> we want column 1
df_2026['raw_probability'] = model.predict_proba(df_2026[features])[:, 1]

print("Predictions generated. Sample:")
display(df_2026[['artist', 'raw_probability']].sort_values('raw_probability', ascending=False).head())

Predictions generated. Sample:


,artist,raw_probability
487,loco & jam,0.897145
345,chapter & verse,0.883882
148,pls&ty,0.879870
832,hugel,0.871595
969,deorro,0.871595


In [14]:
# Apply Logic Controls & Residency Filter
# ---------------------------------------------------------

# Load Residency List
residency_artists = df_residency['artist'].str.lower().str.strip().unique()

def adjust_score(row):
    score = row['raw_probability']
    artist_name = str(row['artist']).lower().strip()
    
    # 1. BOOST: Insomniac Agency
    # Give a 20% boost to Insomniac artists (organizer bias)
    if row['is_insomniac'] == 1:
        score = score * 1.2
        
    # 2. PENALTY: Vegas Residency 2026
    # If they are already booked for a residency, it might conflict (exclusivity clause)
    # Reduce probability by 50%
    has_residency = artist_name in residency_artists
    
    if has_residency:
        score = score * 0.4
        
    # Cap at 1.0 (100%)
    return min(score, 1.0), has_residency

# Apply adjustments
df_2026[['final_probability', 'has_residency']] = df_2026.apply(adjust_score, axis=1, result_type='expand')

# Sort and Show the Lineup Prediction
final_prediction = df_2026.sort_values('final_probability', ascending=False)

print("--- TOP 50 PREDICTED ARTISTS FOR EDC 2026 ---")
cols_output = ['artist', 'final_probability', 'has_residency', 'is_insomniac', 'played_prev_year', 'total_past_appearances']
display(final_prediction[cols_output].head(50))

# Save to CSV
final_prediction.to_csv("../data/result_prediction/edc_2026_prediction.csv", index=False)
print("Saved prediction to ../data/main/edc_2026_prediction.csv")

--- TOP 50 PREDICTED ARTISTS FOR EDC 2026 ---


,artist,final_probability,has_residency,is_insomniac,played_prev_year,total_past_appearances
25,bones,0.964399,False,1,1,4
175,tiësto,0.937713,False,1,1,4
600,decoder,0.933382,False,1,0,0
378,big gigantic,0.931014,False,1,0,0
584,space 92,0.906140,False,1,0,0
487,loco & jam,0.897145,False,0,0,0
345,chapter & verse,0.883882,False,0,0,0
148,pls&ty,0.879870,False,0,0,0
832,hugel,0.871595,False,0,1,4
236,declan james,0.856625,False,1,0,0


Saved prediction to ../data/main/edc_2026_prediction.csv
